In [ ]:
import os
from langchain.chat_models import init_chat_model


model = init_chat_model("claude-sonnet-4-6")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2") #mixedbread-ai/mxbai-embed-large-v1

c:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7198.57it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")

vector_size = len(embeddings.embed_query("sample text"))

if not client.collection_exists("test"):
    client.create_collection(
        collection_name="test",
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
    )
vector_store = QdrantVectorStore(
    client=client,
    collection_name="test",
    embedding=embeddings,
)

In [5]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Total characters: 43047


In [6]:
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 63 sub-documents.


In [8]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['a88372674624466b9a43952a8cae7826', 'cfe5d180203243f5b4e039f5bb8f294b', '7cc3f7e9dee64a5b8100eb8917ccc5b6']


In [9]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [11]:
from langchain.agents import create_agent


tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

In [12]:
query = (
    "What is the standard method for Task Decomposition?\n\n"
    "Once you get the answer, look up common extensions of that method."
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the standard method for Task Decomposition?

Once you get the answer, look up common extensions of that method.
================================== Ai Message ==================================

[{'text': "I'll start by looking up the standard method for Task Decomposition right away!", 'type': 'text'}, {'id': 'toolu_01W3KdyKYQzUn5nafLW4FvGP', 'caller': {'type': 'direct'}, 'input': {'query': 'standard method for Task Decomposition'}, 'name': 'retrieve_context', 'type': 'tool_use'}]
Tool Calls:
  retrieve_context (toolu_01W3KdyKYQzUn5nafLW4FvGP)
 Call ID: toolu_01W3KdyKYQzUn5nafLW4FvGP
  Args:
    query: standard method for Task Decomposition
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 2578, '_id': 'a8a158e0abb54507b3248541f99ee782', '_collection_name

In [ ]:
# from langchain.agents.middleware import dynamic_prompt, ModelRequest

# @dynamic_prompt
# def prompt_with_context(request: ModelRequest) -> str:
#     """Inject context into state messages."""
#     last_query = request.state["messages"][-1].text
#     retrieved_docs = vector_store.similarity_search(last_query)

#     docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

#     system_message = (
#         "You are a helpful assistant. Use the following context in your response:"
#         f"\n\n{docs_content}"
#     )

#     return system_message


# agent = create_agent(model, tools=[], middleware=[prompt_with_context])

In [14]:
query = "What is task decomposition?"
for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is task decomposition?
================================== Ai Message ==================================

Task decomposition is the process of breaking down a complicated task into smaller, more manageable steps or subtasks. It is a key component of planning for AI agents and language models when tackling complex problems.

There are several approaches to task decomposition:

1. **Simple LLM Prompting**: A language model can be guided to decompose tasks using straightforward prompts like "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?" This encourages the model to think through the necessary steps systematically.

2. **Task-Specific Instructions**: Decomposition can be tailored to the nature of the task. For example, asking a model to "Write a story outline" when working on a novel helps break the writing process into structured components.

3. **Human Inputs**: Humans can directly p

In [ ]:
# Deepseek 70B 

